# Capítulo 1 · Por qué fallan los agentes: el banco de trabajo

Un agente de programación en el repositorio de la tienda Ciclos Norte: trece módulos de Python, 59 pruebas y catorce tareas escritas como las escribiría alguien de la tienda, cada una con pruebas ocultas que el agente no ve. Primero, el agente solo con la petición. Después, con un banco de trabajo mínimo: un `AGENTS.md` con la definición de terminado y una puerta que la comprueba antes de aceptar un «terminado».

**Antes de empezar, activa la GPU:** menú *Entorno de ejecución* → *Cambiar tipo de entorno de ejecución* → *GPU T4* → *Guardar*.

La explicación completa está en el capítulo en PDF. Este cuaderno es el laboratorio.

## 1. Cargar el modelo, el repositorio y las herramientas

El modelo ocupa unos 8 GB; la primera vez tarda unos minutos en descargarse. `preparar` descarga del repositorio del libro el programa de la tienda, las tareas y el módulo de las herramientas.

In [ ]:
!pip install -q transformers==5.19.0  # las versiones con las que se comprobó el libro

In [ ]:
"""Capítulo 1: por qué fallan los agentes y el banco de trabajo mínimo."""
import importlib
import io
import json
import re
import sys
import time
import urllib.request
import zipfile
from pathlib import Path

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODELO_GPU = "Qwen/Qwen3-4B-Instruct-2507"  # con la GPU T4 gratuita de Colab
MODELO_CPU = "Qwen/Qwen3-1.7B"              # sin GPU: más pequeño, y peor agente
RAIZ = Path.cwd()                           # donde están datos/ y comun_libro/
REPOSITORIO = ("https://github.com/HernanDiaz/ai-agents-real-projects/"
               "archive/refs/heads/main.zip")
TAREAS = ["T01", "T02", "T03", "T04", "T05", "T10", "T11", "T13", "T14", "T15",
          "T17", "T18", "T19", "T20"]       # las tareas de una sola sesión
MAX_PASOS = 30             # llamadas al modelo por tarea
MAX_TOKENS_PASO = 600      # tokens nuevos por llamada
LIMITE_ENTRADA = 12_000    # tokens de la conversación: más no cabe en la T4
RESULTADOS_INTACTOS = 2    # los últimos resultados de herramientas no se recortan
MAX_RECHAZOS = 3           # veces que la puerta devuelve el trabajo
hr = None                  # el módulo de herramientas, tras preparar()



def cargar(nombre=None):
    """Carga el modelo y su tokenizador: en la GPU, en float16; si no, en la CPU."""
    gpu = torch.cuda.is_available()
    if nombre is None:
        nombre = MODELO_GPU if gpu else MODELO_CPU
    tok = AutoTokenizer.from_pretrained(nombre)
    modelo = AutoModelForCausalLM.from_pretrained(
        nombre, dtype=torch.float16 if gpu else torch.float32)
    modelo.to("cuda" if gpu else "cpu").eval()
    print(f"Modelo {nombre} en {'cuda' if gpu else 'cpu'}")
    return tok, modelo


def preparar():
    """El repositorio de Ciclos Norte, sus tareas y las herramientas del agente.
    Si no están (en Colab), se descargan del repositorio del libro."""
    global hr
    if not (RAIZ / "comun_libro" / "herramientas_repo.py").exists():
        with urllib.request.urlopen(REPOSITORIO) as respuesta:
            archivo = zipfile.ZipFile(io.BytesIO(respuesta.read()))
        for nombre in archivo.namelist():
            relativa = nombre.split("/", 1)[1]
            if relativa.startswith(("datos/", "comun_libro/")) and not nombre.endswith("/"):
                destino = RAIZ / relativa
                destino.parent.mkdir(parents=True, exist_ok=True)
                destino.write_bytes(archivo.read(nombre))
    sys.path.insert(0, str(RAIZ / "comun_libro"))
    hr = importlib.import_module("herramientas_repo")
    print(f"Repositorio en {hr.REPO}: {len(hr.archivos_de(hr.REPO))} archivos")

In [ ]:
tok, modelo = cargar()
preparar()

## 2. Las herramientas, sin modelo

Una copia del repositorio que se borra al salir del `with`, y las herramientas que usará el agente. Fíjate en el error de `editar` cuando el trozo no coincide, y en cómo rechaza una edición que rompería el archivo.

In [ ]:
with hr.Espacio() as espacio:
    h = hr.Herramientas(espacio)
    print(h.llamar("buscar", {"patron": r"def zona", "carpeta": "envios.py"}))
    print(h.llamar("leer_archivo", {"ruta": "envios.py", "desde": 6, "hasta": 15}))
    print(h.llamar("editar", {"ruta": "envios.py", "buscar": "codigo = str(codigo_postal)",
                              "poner": "  codigo = str(codigo_postal)"}))
    print(h.llamar("ejecutar_pruebas")["resultado"])

Una tarea por dentro: la petición, el alcance y una de sus pruebas ocultas.

In [ ]:
tarea = hr.leer_tarea("T02")
print(tarea["peticion"])
print(tarea["alcance"])
print(sorted(p.name for p in (tarea["carpeta"] / "pruebas_ocultas").iterdir()))

## 3. El modelo

La plantilla con las herramientas, la generación, la lectura de las llamadas y el recorte de la conversación cuando no cabe en la GPU.

In [ ]:
def plantilla(tok, mensajes):
    """La conversación como texto, con las herramientas, en la plantilla del modelo."""
    return tok.apply_chat_template(mensajes, tools=hr.ESQUEMAS, tokenize=False,
                                   add_generation_prompt=True)


def generar(tok, modelo, texto):
    """Una respuesta del modelo, con decodificación voraz. Devuelve el texto y los
    tokens de entrada y de salida."""
    entrada = tok(texto, return_tensors="pt").to(modelo.device)
    with torch.no_grad():
        salida = modelo.generate(**entrada, max_new_tokens=MAX_TOKENS_PASO,
                                 do_sample=False, temperature=None, top_p=None,
                                 top_k=None, pad_token_id=tok.eos_token_id)
    n_entrada = entrada["input_ids"].shape[1]
    nuevos = salida[0, n_entrada:].cpu()
    del entrada, salida
    if torch.cuda.is_available():
        torch.cuda.empty_cache()  # cada paso tiene otra longitud: sin esto la
        #                           memoria reservada crece hasta llenar la GPU
    return tok.decode(nuevos, skip_special_tokens=True).strip(), n_entrada, len(nuevos)


def leer_llamadas(respuesta):
    """Las llamadas {name, arguments} que el modelo escribe entre <tool_call> y
    </tool_call>. Una llamada que no se entiende queda con name None."""
    llamadas = []
    for bloque in re.findall(r"<tool_call>(.*?)(?:</tool_call>|$)", respuesta, re.S):
        try:
            datos = json.loads(bloque.strip(), strict=False)
        except ValueError:
            datos = None
        if isinstance(datos, dict) and "name" in datos:
            argumentos = datos.get("arguments", {})
            llamadas.append({"name": datos["name"],
                             "arguments": argumentos if isinstance(argumentos, dict)
                             else {}})
        else:
            llamadas.append({"name": None, "arguments": {}})
    return llamadas


def recortar_conversacion(tok, mensajes):
    """Si la conversación pasa de LIMITE_ENTRADA tokens, cambia los resultados de
    herramientas más antiguos por un aviso corto. Devuelve el texto para el modelo y
    cuántos ha recortado (None si ni así cabe)."""
    texto = plantilla(tok, mensajes)
    recortados = 0
    candidatos = [m for m in mensajes if m["role"] == "tool"
                  and not m.get("recortado")][:-RESULTADOS_INTACTOS or None]
    while len(tok(texto)["input_ids"]) > LIMITE_ENTRADA:
        if not candidatos:
            return texto, None
        mensaje = candidatos.pop(0)
        mensaje["content"] = json.dumps(
            {"aviso": "Resultado antiguo recortado para ahorrar memoria; vuelve a "
                      "pedirlo si lo necesitas.",
             "inicio": mensaje["content"][:150]}, ensure_ascii=False)
        mensaje["recortado"] = True
        recortados += 1
        texto = plantilla(tok, mensajes)
    return texto, recortados

## 4. Las instrucciones y la puerta

In [ ]:
SISTEMA = (
    "Eres un agente de programación que trabaja en el repositorio de Ciclos Norte, "
    "una tienda de bicicletas de Oviedo. Es un programa de gestión en Python con "
    "varios módulos (pedidos, devoluciones, existencias, facturas, avisos…), sus "
    "pruebas en la carpeta tests/ y unos datos de ejemplo. Recibirás una petición de "
    "alguien de la tienda y tienes que resolverla cambiando el código con las "
    "herramientas: listar archivos, leer por rangos de líneas, buscar, editar por "
    "sustitución exacta, ejecutar las pruebas y unas pocas órdenes permitidas. "
    "Trabaja paso a paso, con una herramienta cada vez. Cuando hayas terminado, "
    "responde sin llamar a ninguna herramienta, empezando por «TERMINADO:» y "
    "explicando en una o dos frases qué has cambiado. Si no puedes terminar, empieza "
    "por «NO TERMINADO:» y explica por qué.")


AGENTS_MD = """# AGENTS.md · Ciclos Norte

## Al empezar
1. Lee la petición y escribe en una frase qué comportamiento hay que cambiar.
2. Busca el código que lo decide (buscar) y léelo antes de editar.

## Definición de terminado
- Has cambiado el código que decide ese comportamiento (no solo comentarios).
- Has añadido una prueba que comprueba lo que pide la petición, en el archivo de
  pruebas del módulo que cambias (tests/test_envios.py para envios.py, etc.).
- Has ejecutado ejecutar_pruebas después de tu último cambio y pasan todas.
Solo entonces respondes «TERMINADO:». Si una prueba falla, corrige y vuelve a ejecutarlas.

## Cómo añadir una prueba
Todos los archivos de tests/ acaban con `if __name__ == "__main__":`. Edita
sustituyendo esa línea por una clase nueva seguida de la misma línea:

    class PruebasTarea(unittest.TestCase):
        def test_lo_pedido(self):
            self.assertEqual(..., ...)


    if __name__ == "__main__":

Importa dentro de la prueba lo que necesites (from envios import gastos_envio).

## No toques
- datos/ y docs/ salvo que la petición lo pida.
- Las pruebas que ya existen: añade las tuyas, no cambies las de otros.
"""


def dice_terminado(respuesta):
    """True si la respuesta final dice que la tarea está hecha. Una respuesta vacía
    no dice nada."""
    if not respuesta.strip():
        return False
    if re.search(r"NO\s+TERMINADO", respuesta, re.I):
        return False
    if re.search(r"TERMINADO", respuesta):
        return True
    return not re.search(r"no (he podido|puedo|consigo|es posible)|necesito|\?",
                         respuesta, re.I)


def que_falta(registro):
    """Lo que falta para la definición de terminado, según el registro de llamadas:
    un cambio en el código, una prueba nueva y las pruebas pasadas después."""
    ediciones = [i for i, r in enumerate(registro)
                 if r["herramienta"] in ("editar", "crear_archivo") and r["ok"]]
    ruta = lambda i: registro[i]["argumentos"].get("ruta", "")  # noqa: E731
    falta = []
    if not any(not ruta(i).startswith("tests/") for i in ediciones):
        falta.append("no has cambiado el código del programa")
    if not any(ruta(i).startswith("tests/") for i in ediciones):
        falta.append("no has añadido ninguna prueba en tests/")
    ultima = ediciones[-1] if ediciones else -1
    pruebas = [r for r in registro[ultima + 1:] if r["herramienta"] == "ejecutar_pruebas"]
    if not pruebas:
        falta.append("no has ejecutado ejecutar_pruebas después de tu último cambio")
    elif not pruebas[-1]["resultado"].get("todas_pasan"):
        falta.append("la última ejecución de las pruebas tiene fallos")
    return falta

## 5. El agente

El bucle del libro anterior, con el `AGENTS.md` opcional y la puerta. La tarea de los códigos postales, primero solo con la petición y después con el banco mínimo:

In [ ]:
def agente(tok, modelo, tarea, instrucciones="", puerta=False, max_pasos=MAX_PASOS,
           ver=False):
    """El bucle del agente sobre una copia limpia del repositorio. Con
    «instrucciones», el AGENTS.md va tras el mensaje de sistema; con «puerta», un
    «TERMINADO» sin la definición de terminado se devuelve con lo que falta.
    Devuelve la fila de resultados, ya evaluada con las pruebas ocultas."""
    inicio = time.time()
    sistema = SISTEMA + ("\n\n" + instrucciones if instrucciones else "")
    mensajes = [{"role": "system", "content": sistema},
                {"role": "user", "content": tarea["peticion"]}]
    final, pasos, entrada, salida, rechazos = None, 0, 0, 0, 0
    with hr.Espacio() as espacio:
        herramientas = hr.Herramientas(espacio)
        for pasos in range(1, max_pasos + 1):
            texto, recortados = recortar_conversacion(tok, mensajes)
            if recortados is None:
                break
            respuesta, n_entrada, n_salida = generar(tok, modelo, texto)
            entrada, salida = entrada + n_entrada, salida + n_salida
            llamadas = leer_llamadas(respuesta)
            if not llamadas:
                mensajes.append({"role": "assistant", "content": respuesta})
                if ver:
                    print(f"[{pasos}] {respuesta}")
                falta = que_falta(herramientas.registro) if puerta else []
                if dice_terminado(respuesta) and falta and rechazos < MAX_RECHAZOS:
                    rechazos += 1
                    aviso = ("Todavía no está terminado según AGENTS.md: "
                             + "; ".join(falta) + ". Sigue trabajando.")
                    mensajes.append({"role": "user", "content": aviso})
                    if ver:
                        print(f"    PUERTA: {aviso}")
                    continue
                final = respuesta
                break
            pensamiento = respuesta.split("<tool_call>")[0].strip()
            mensajes.append({"role": "assistant", "content": pensamiento,
                             "tool_calls": [{"type": "function", "function": l}
                                            for l in llamadas if l["name"]]})
            for llamada in llamadas:
                if llamada["name"] is None:
                    resultado = {"error": "No se entiende la llamada a la herramienta."}
                else:
                    resultado = herramientas.llamar(llamada["name"], llamada["arguments"])
                if ver:
                    print(f"[{pasos}] {llamada['name']}"
                          f"({json.dumps(llamada['arguments'], ensure_ascii=False)[:90]})"
                          f" -> {json.dumps(resultado, ensure_ascii=False)[:110]}")
                mensajes.append({"role": "tool", "name": llamada["name"] or "?",
                                 "content": json.dumps(resultado, ensure_ascii=False)})
        evaluacion = hr.evaluar_intento(espacio, tarea)
        falta = que_falta(herramientas.registro)
    terminado = final is not None and dice_terminado(final)
    return {"tarea": tarea["id"], "tipo": tarea["meta"]["tipo"],
            "resuelta": evaluacion["resuelta"], "dice_terminado": terminado,
            "terminado_falso": terminado and not evaluacion["resuelta"],
            "cumple_definicion": not falta, "rechazos": rechazos,
            "ocultas": evaluacion["ocultas"], "visibles": evaluacion["visibles"],
            "fuera_de_alcance": evaluacion["fuera_de_alcance"],
            "edito": any(r["herramienta"] in ("editar", "crear_archivo") and r["ok"]
                         for r in herramientas.registro),
            "ejecuto_pruebas": any(r["herramienta"] == "ejecutar_pruebas"
                                   for r in herramientas.registro),
            "pasos": pasos, "tokens_entrada": entrada, "tokens_salida": salida,
            "segundos": round(time.time() - inicio, 1),
            "respuesta": final or "(sin respuesta final)",
            "llamadas": [[r["herramienta"], r["argumentos"].get("ruta", ""), r["ok"]]
                         for r in herramientas.registro],
            "diff": evaluacion["diff"]}

In [ ]:
for nombre, opciones in (("SOLO LA PETICIÓN", {}),
                         ("BANCO MÍNIMO", {"instrucciones": AGENTS_MD, "puerta": True})):
    print(f"--- {nombre}")
    fila = agente(tok, modelo, hr.leer_tarea("T02"), ver=True, **opciones)
    print("Resuelta:", fila["resuelta"], "· Dice terminado:", fila["dice_terminado"])
    print(fila["diff"][:1500] or "(sin cambios)")

## 6. Las catorce tareas con las dos variantes

Tarda algo más de una hora en una GPU de escritorio y bastante más en Colab. Para una primera prueba, pasa `tareas=TAREAS[:4]` a `probar`.

In [ ]:
VARIANTES = {
    "solo la petición": {},
    "banco mínimo": {"instrucciones": AGENTS_MD, "puerta": True},
}


def resumen(filas):
    """Los totales de una variante."""
    return {"tareas": len(filas),
            "resueltas": sum(f["resuelta"] for f in filas),
            "dice_terminado": sum(f["dice_terminado"] for f in filas),
            "terminado_falso": sum(f["terminado_falso"] for f in filas),
            "editan": sum(f["edito"] for f in filas),
            "ejecutan_pruebas": sum(f["ejecuto_pruebas"] for f in filas),
            "con_fuera_de_alcance": sum(bool(f["fuera_de_alcance"]) for f in filas),
            "pasos": sum(f["pasos"] for f in filas),
            "tokens_entrada": sum(f["tokens_entrada"] for f in filas),
            "tokens_salida": sum(f["tokens_salida"] for f in filas),
            "segundos": round(sum(f["segundos"] for f in filas), 1)}


def probar(tok, modelo, opciones, tareas=TAREAS):
    """Una variante con todas las tareas: sus filas y su resumen."""
    filas = []
    for identificador in tareas:
        fila = agente(tok, modelo, hr.leer_tarea(identificador), **opciones)
        filas.append(fila)
        print(f"  {identificador} {'resuelta' if fila['resuelta'] else 'no      '}"
              f"  {'dice terminado' if fila['dice_terminado'] else 'no lo dice    '}"
              f"  pasos={fila['pasos']:2d}  {fila['segundos']:6.1f} s", flush=True)
    return {"filas": filas, "resumen": resumen(filas)}

In [ ]:
resultados = {}
for nombre, opciones in VARIANTES.items():
    print(f"== {nombre}")
    resultados[nombre] = probar(tok, modelo, opciones)
    print(resultados[nombre]["resumen"])

Los «terminado» falsos del banco mínimo: la respuesta, las llamadas y el diff. ¿Qué cambió de verdad?

In [ ]:
for f in resultados["banco mínimo"]["filas"]:
    if f["terminado_falso"]:
        print("✗", f["tarea"], f["respuesta"][:200])
        print("   ", [(h, ruta, ok) for h, ruta, ok in f["llamadas"] if h in ("editar", "crear_archivo")])
        print(f["diff"][:800])

## Ejercicios

Los archivos de resultados que citan los ejercicios están en `datos/resultados/capitulo_1/`, que `preparar` descarga con los datos.

1. Para cada uno de los diez «terminado» falsos del agente solo con la petición, di qué superficie del banco de trabajo le faltaba. ¿Hay alguno que no encaje en ninguna de las siete? *(Sin GPU.)*

2. Ejecuta la tarea de los códigos postales (T02) con `ver=True` en las dos variantes. ¿En qué paso se separan las dos trazas? ¿Qué hace el agente con los rechazos de la puerta? *(GPU, unos 20 minutos.)*

3. Prueba el `AGENTS.md` sin la puerta, con las catorce tareas. ¿Bastan las instrucciones para que el agente ejecute las pruebas? ¿Cuántos «terminado» falsos quedan? Los resultados con las catorce tareas ya están en `resultados_ejercicios.json`; para verlo tú, basta con tres, por ejemplo T01, T04 y T13. *(Sin GPU con los resultados guardados; GPU, unos 30 minutos, con esas tareas.)*

4. Prueba ahora la puerta sin el `AGENTS.md`. El agente no conoce la definición de terminado hasta que la puerta le devuelve el trabajo. ¿Qué pieza aporta más? Los resultados con las catorce tareas ya están en `resultados_ejercicios.json`; para verlo tú, basta con tres, por ejemplo T01, T04 y T13. *(Sin GPU con los resultados guardados; GPU, unos 30 minutos, con esas tareas.)*

5. Tres tareas de una sola sesión no entraron en el banco: las transferencias canceladas ya pagadas (T06), los seis meses de revisión gratuita (T07) y las promociones sin consentimiento (T08). Lee sus peticiones y ejecútalas con el banco mínimo. ¿Se parecen sus resultados a los de las otras? Los resultados también están en `resultados_ejercicios.json`. *(GPU, unos 30 minutos, o sin GPU con los resultados guardados.)*

6. Lee la función que escribió el agente para la tarea de las existencias bajas y las pruebas ocultas de esa tarea, que están en la carpeta `datos/tareas/T13/pruebas_ocultas`. ¿Qué frase de la petición no cumple? Escribe una prueba que lo detecte. *(Sin GPU.)*

7. Cambia la herramienta `editar` para que rechace las ediciones en las que `buscar` y `poner` son iguales, y repite las tres tareas en las que una edición no cambiaba nada: las devoluciones parciales, Portugal y Andorra y las existencias bajas (T03, T10 y T13). ¿Siguen pasando la puerta? Los resultados también están en `resultados_ejercicios.json`. *(GPU, unos 30 minutos, o sin GPU con los resultados guardados.)*

8. Añade a la definición de terminado una línea que pida releer la petición frase a frase y comprobar cada frase con la prueba. Repite las mismas tres tareas. ¿Ayuda una instrucción así, o hace falta algo que la compruebe? Los resultados también están en `resultados_ejercicios.json`. *(GPU, unos 30 minutos, o sin GPU con los resultados guardados.)*

9. Con los resultados del capítulo, calcula los tokens de entrada y los segundos por tarea resuelta con cada variante. Si cada millón de tokens de entrada costara 1 euro y cada «terminado» falso costara 20 euros de trabajo de una persona, ¿qué variante sale más barata? *(Sin GPU.)*

10. **Proyecto.** Escribe un `AGENTS.md` para un repositorio tuyo, con una definición de terminado que se pueda comprobar, y una puerta que mire el diff en lugar del registro: debe exigir cambios fuera de `tests/` que no sean solo comentarios y una prueba nueva que falle con el código original y pase con el nuevo. Pruébala con tres tareas de ese repositorio.

## Referencias

Modelo [Qwen3-4B-Instruct-2507](https://huggingface.co/Qwen/Qwen3-4B-Instruct-2507). [SWE-agent](https://arxiv.org/abs/2405.15793) (Yang y colaboradores, 2024). [SWE-bench](https://arxiv.org/abs/2310.06770) (Jiménez y colaboradores, 2024). La convención [AGENTS.md](https://agents.md). [Effective harnesses for long-running agents](https://www.anthropic.com/engineering/effective-harnesses-for-long-running-agents) (Anthropic, 2025). La tienda Ciclos Norte, su programa y sus tareas son ficticios. Código y experimentos propios.